# Forecasts figures

Use the bao kernel in VS Code in your allocated compute session. Choose saved inputs below, then edit and run the plot cells you need. Leave an input as `None` to skip that section.


In [ ]:
from pathlib import Path
import csv
import json
import os
import sys
import itertools

import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

REPO = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pzbao/run.py").is_file())
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

from pzbao.figures import data as readers, style

BINS, TIERS, STATES, PARAMETERS = readers.BINS, readers.TIERS, readers.STATES, readers.PARAMETERS
LABELS, ROBUSTNESS = readers.LABELS, readers.ROBUSTNESS
figures = {}


def show(fig, name, caption=""):
    figures[name] = fig
    display(fig)
    if caption:
        display(Markdown(caption))
    plt.close(fig)


In [ ]:
TIER_COLORS = ("#4d4d4d", "#1f77b4", "#2ca02c", "#ff7f00", "#e41a1c")
# Colors read from the SVG of Paillas et al. 2404.03005v3, Figure 15.
BIN_COLORS = dict(zip(BINS, ("#9acd32", "#9acd32", "#ffa500", "#ff4500", "#b22222")))
ZLABELS = (
    r"$0.1\leq z<0.2$",
    r"$0.2\leq z<0.4$",
    r"$0.4\leq z<0.6$",
    r"$0.6\leq z<0.8$",
    r"$0.8\leq z<1.0$",
)
VARIANTS = ("baseline", "pre", "pk", "ngc", "flat", "pcs2", "reciso", "r25")
VARIANT_LABELS = (
    "Baseline: power3, RecSym, R15",
    "Pre-reconstruction",
    "Power spectrum only",
    "NGC only",
    "Flat damping priors",
    "PCS2 spline broadband",
    "RecIso",
    r"$R=25\,h^{-1}\mathrm{Mpc}$",
)
LABELS = {"qiso": r"$\alpha_{\rm iso}$", "qap": r"$\alpha_{\rm AP}$"}


def presentation_style():
    import matplotlib as mpl

    mpl.rcParams.update(
        {
            "font.family": "serif",
            "mathtext.fontset": "cm",
            "font.size": 11,
            "axes.labelsize": 13,
            "axes.titlesize": 12,
            "axes.linewidth": 1.2,
            "xtick.direction": "in",
            "ytick.direction": "in",
            "xtick.top": True,
            "ytick.right": True,
            "savefig.dpi": 200,
        }
    )


def panel_grid():
    import matplotlib.pyplot as plt

    fig, axes = plt.subplots(2, 3, figsize=(13.5, 8), layout="constrained")
    axes = list(axes.flat)
    axes[-1].axis("off")
    return fig, axes


presentation_style()

from pzbao.figures.style import sigma_color, two_col
from pzbao.diagnostics import covariance as forecast


In [ ]:
DENSITY_CAPTION = (
    "Conditional power3 Gaussian forecast, with all nuisance parameters marginalized. "
    "Each pre/post state is normalized by its own current-density FoM. Ordered contributions: "
    "current = 1, target n(z) including the modelled transverse reconstruction response, "
    "then area to f_sky = {target_fsky:g} with unchanged priors. Signed density contributions are retained. "
    "Native redshifts have no density target; 0.001 is a constructed quarter-density proxy. "
    "Fixed reference window and effective redshift; radial reconstruction response is not "
    "independently calibrated. Crosses mark flagged or unavailable source profiles. All source posterior ESS "
    "estimates remain unavailable. These are local model forecasts, not measured survey gains."
)


In [ ]:
PRECISION_CAPTION = (
    "Direct fractional sigma_R/R for the four photometric mock tiers, plus a "
    "DESI DR1 density/volume forecast and the mock-footprint cosmic-variance reference. "
    "Mock points use effective redshift; DR1 uses bin centres. Effective volumes sum the "
    "attained cap densities and physical cap volumes, with the same frozen tracer-specific "
    "P0 in both mock and DR1 calculations. No common-area rescaling is applied. The original "
    "factor of 100 expressed percentages; every plotted value is now a dimensionless fraction. "
    "R is the combined distance-scale precision calibrated from DESI Table 7. The 0.001 tier "
    "is a constructed quarter-density proxy. This effective-volume diagnostic omits anisotropic "
    "photo-z information loss and reconstruction response; it is not fitted posterior precision."
)


## Conditional density and area forecast

Choose a saved `forecast-density-area.json`. These are conditional Gaussian forecasts; signed contributions, source-fit flags and sampling limits remain visible.


In [ ]:
def density_plot_forecast(rows):
    from matplotlib.patches import Patch
    from matplotlib.ticker import MaxNLocator

    presentation_style()
    (target_fsky,) = {r["target_fsky"] for r in rows}
    fig, axes = panel_grid()
    for ax, name, zlabel in zip(axes, forecast.BINS, ZLABELS):
        selected = {
            (r["route"]["condition"], r["route"]["reconstruction_state"]): r
            for r in rows
            if r["route"]["measurement_bin"] == name
        }
        endpoints = [0.0, 1.0]
        for x, tier in enumerate(forecast.TIERS):
            for offset, state, hatch in (
                (-0.18, forecast.STATES[0], "///"),
                (0.18, forecast.STATES[1], None),
            ):
                row = selected[(tier, state)]
                center = x + offset
                rn, rt = row["density_ratio"], row["total_ratio"]
                ax.bar(center, 1.0, 0.325, color=".85", edgecolor=".25", linewidth=0.6, hatch=hatch)
                ax.bar(
                    center,
                    rn - 1.0,
                    0.325,
                    bottom=1.0,
                    color=BIN_COLORS[name],
                    edgecolor=".25",
                    linewidth=0.6,
                    hatch=hatch,
                )
                ax.bar(
                    center,
                    rt - rn,
                    0.325,
                    bottom=rn,
                    color="#6d87a6",
                    edgecolor=".25",
                    linewidth=0.6,
                    hatch=hatch,
                )
                if rn < 1.0:
                    ax.bar(
                        center,
                        rn - 1.0,
                        0.325,
                        bottom=1.0,
                        facecolor="none",
                        edgecolor=BIN_COLORS[name],
                        linewidth=1.2,
                        hatch="xx",
                    )
                    ax.annotate(
                        "",
                        (center, rn),
                        (center, 1.0),
                        arrowprops=dict(arrowstyle="->", lw=0.8, color="black"),
                    )
                # A negative density increment is drawn down from unity, with its endpoint visible.
                ax.plot([center - 0.16, center + 0.16], [rn, rn], color="black", lw=0.8)
                if readers.flagged(row):
                    ax.plot(center, rt, "kx", ms=4)
                endpoints.extend((rn, rt))
        ax.set(
            xticks=range(5),
            xticklabels=("native*", ".001†", ".003", ".01", ".03"),
            xlabel=r"$\sigma_z/(1+z)$",
            ylabel=r"FoM / current FoM",
            title=f"{name}  {zlabel}",
            xlim=(-0.6, 4.6),
        )
        span = max(endpoints) - min(endpoints)
        ax.set_ylim(min(endpoints), max(endpoints) + 0.12 * span)
        ax.axhline(1.0, color=".4", lw=0.7, ls=":")
        ax.yaxis.set_major_locator(MaxNLocator(5))
        ax.set_axisbelow(True)
        ax.grid(axis="y", color=".9")
    handles = [
        Patch(facecolor=".85", edgecolor=".25", label="Current density and area"),
        Patch(facecolor="#ffa500", edgecolor=".25", label="Target n(z): noise + response"),
        Patch(
            facecolor="#6d87a6",
            edgecolor=".25",
            label=rf"Then area to $f_{{\rm sky}}={target_fsky:g}$",
        ),
        Patch(facecolor="white", edgecolor=".25", hatch="///", label="Pre-reconstruction (left)"),
        Patch(facecolor="white", edgecolor=".25", label="Post-reconstruction (right)"),
    ]
    axes[-1].legend(handles=handles, loc="upper left", frameon=False, fontsize=10)
    axes[-1].text(
        0.02,
        0.43,
        "* Native: area only; no density target\n"
        "† Constructed quarter-density proxy\n"
        "Density colors follow the tracer palette\n"
        "Black tick: endpoint after density change\n"
        "Cross-hatch: negative density increment\n"
        "× Source profile flagged or unavailable\n\n"
        "Fixed window and effective redshift\n"
        "Transverse reconstruction response only\n"
        "Conditional Gaussian model; not a posterior",
        transform=axes[-1].transAxes,
        va="top",
        fontsize=9,
    )
    fig.suptitle(
        "Conditional power3 forecast: density, reconstruction response and area", fontsize=14
    )
    show(fig, "forecast-density-area", DENSITY_CAPTION.format(target_fsky=target_fsky))
    return fig


DENSITY_FORECAST = None
if DENSITY_FORECAST is not None:
    forecast_rows = readers.load_density_forecast(DENSITY_FORECAST)
    density_plot_forecast(forecast_rows)


## Fractional precision and DESI DR1

Choose the directory containing `precision-values.json`. The plotted quantity is a density/volume forecast. The standalone DESI DR1 reader is available for further comparisons.


In [ ]:
def precision_plot_precision(rows, *, name="sigmaR-over-R-vs-z"):
    import matplotlib.pyplot as plt
    from matplotlib.ticker import MaxNLocator, ScalarFormatter

    presentation_style()
    fig, ax = plt.subplots(figsize=(7.0, 3.7), layout="constrained")
    tiers = ("sigma_0.001", "sigma_0.003", "sigma_0.01", "sigma_0.03")  # the unsmeared control is not a sample
    labels = (
        r"Mock $\sigma_z/(1+z)=0.001$",
        r"Mock $\sigma_z/(1+z)=0.003$",
        r"Mock $\sigma_z/(1+z)=0.01$",
        r"Mock $\sigma_z/(1+z)=0.03$",
    )

    def selected(condition):
        group = {r["measurement_bin"]: r for r in rows if r["condition"] == condition}
        return [group[name] for name in readers.BINS]

    for tier, label, color, marker, line in zip(
        tiers,
        labels,
        [style.sigma_color(s) for s in (.001, .003, .01, .03)],
        ("s", "^", "v", "D"),
        ("--", "-.", ":", (0, (5, 2, 1, 2))),
    ):
        group = selected(tier)
        ax.plot(
            [r["z"] for r in group],
            [r["sigma_r_over_r"] for r in group],
            color=color,
            marker=marker,
            linestyle=line,
            lw=1.6,
            ms=5,
            markeredgecolor="black",
            markeredgewidth=0.6,
            label=label,
        )
    group = selected("desi_dr1")
    ax.plot(
        [r["z"] for r in group],
        [r["sigma_r_over_r"] for r in group],
        color="black",  # distinct from every tier colour; the cosmic-variance line is dotted
        marker="x",
        ls="--",
        lw=1.6,
        ms=6,
        label="DESI DR1 (density/volume)",
    )
    footprint = selected("sigma_0.003")  # cosmic variance depends on the footprint volume only
    ax.plot(
        [r["z"] for r in footprint],
        [r["cosmic_variance_sigma_r_over_r"] for r in footprint],
        color="black",
        ls=":",
        lw=1.5,
        label="Cosmic variance (mock footprint)",
    )
    ax.set(
        xlabel="Redshift z",
        ylabel=r"$\sigma_R/R$",
        xlim=(0.1, 1.0),
        ylim=(0, max(r["sigma_r_over_r"] for r in rows) * 1.12),
    )
    ax.set_xticks((0.15, 0.3, 0.5, 0.7, 0.9))
    ax.yaxis.set_major_locator(MaxNLocator(6))
    ax.yaxis.set_major_formatter(ScalarFormatter(useOffset=False))
    ax.grid(axis="y", color=".9")
    ax.set_axisbelow(True)
    ax.minorticks_on()
    ax.legend(ncol=3, fontsize=8, frameon=False, loc="lower left", bbox_to_anchor=(0, 1.02))
    show(fig, name, PRECISION_CAPTION)
    return fig


PRECISION_REPORT = None
if PRECISION_REPORT is not None:
    bundle = readers.read_tables(PRECISION_REPORT, ["precision-values.json"])
    precision = bundle["precision-values.json"]
    precision_plot_precision(precision["rows"])


In [ ]:
from pzbao.figures.style import PRD_STYLE_FILE, categorical_color, one_col
from pzbao.figures.data import SCENARIO_MARKERS, SCENARIO_LINESTYLES


def plot_r_precision(projection):
    """Render the reviewed single-panel survey precision forecast."""
    import matplotlib.pyplot as plt

    plt.style.use(PRD_STYLE_FILE)
    fig, ax = plt.subplots(figsize=one_col(h=3.25))
    redshift = np.asarray([row["redshift_center"] for row in projection["bins"]], dtype=float)
    for index, scenario in enumerate(projection["scenarios"]):
        ax.plot(
            redshift,
            scenario["sigma_r_over_r_pct"],
            color=categorical_color(index),
            marker=SCENARIO_MARKERS[index],
            linestyle=SCENARIO_LINESTYLES[index],
            label=scenario["label"],
        )
    cosmic = projection["spherex_cosmic_variance"]
    ax.plot(
        redshift,
        cosmic["sigma_r_over_r_pct"],
        color="black",
        marker="D",
        linestyle="-",
        linewidth=1.2,
        label=cosmic["label"],
    )
    ax.set_xlabel(r"redshift-bin center $z$")
    ax.set_xticks(redshift)
    ax.grid(True, alpha=0.25)
    ax.set_ylabel(r"forecast precision $100\sigma_R/R$ [%]")
    ax.set_ylim(0.0, 2.0)
    ax.legend(
        frameon=False,
        fontsize="xx-small",
        ncol=2,
        loc="lower center",
        bbox_to_anchor=(0.5, 1.01),
        columnspacing=0.8,
        handletextpad=0.4,
    )
    ax.text(
        0.985,
        0.965,
        r"$100\sigma_R/R=A(z)/\sqrt{V_{\rm eff}}$",
        transform=ax.transAxes,
        ha="right",
        va="top",
        fontsize="x-small",
    )
    fig.tight_layout(pad=0.4)
    show(fig, "fig_survey_r_precision")
    return fig


SURVEY_VOLUME_DATA = None
if SURVEY_VOLUME_DATA is not None:
    projection = readers.build_r_precision_projection(readers.read_json(SURVEY_VOLUME_DATA))
    plot_r_precision(projection)


In [ ]:
def claim_figures_plot_selection(density, radial, area, tiers):
    import matplotlib.pyplot as plt

    fig, axes = plt.subplots(1, 5, figsize=(13, 3.1), layout="constrained")
    for ax, bin_name in zip(axes, BINS):
        group = {r["condition"]: r for r in density if r["measurement_bin"] == bin_name}
        x = np.arange(len(tiers))
        ax.plot(x, [group[t]["attained_nbar_h3_mpc3"] for t in tiers], "o-", label="attained")
        ax.plot(
            x, [group[t]["target_nbar_h3_mpc3"] or np.nan for t in tiers], "x--", label="target"
        )
        ax.set(title=bin_name, yscale="log")
        ax.set_xticks(x, [t.removeprefix("sigma_") for t in tiers], rotation=45, fontsize=8)
    axes[0].set_ylabel(r"$\bar n\ [h^3\,\mathrm{Mpc}^{-3}]$")
    axes[-1].legend(fontsize=8)
    show(
        fig,
        "selection-density",
        "Attained counts divided by cap-summed comoving volumes. The 0.001 tier is a "
        "constructed quarter-density proxy. The target's 0-0.2 shell approximates "
        "BGS1's 0.1-0.2 shell.",
    )
    fig, axes = plt.subplots(1, 5, figsize=(13, 2.9), layout="constrained")
    for ax, bin_name in zip(axes, BINS):
        row = next(r for r in radial if r["measurement_bin"] == bin_name)
        for field, label, style in (
            ("attained_pdf", "attained", "-"),
            ("target_pdf", "shell target", "--"),
        ):
            ax.stairs(row[field], row["z_edges"], label=label, ls=style)
        ax.set(title=bin_name, xlabel="z")
    axes[0].set_ylabel("Normalized dN/dz")
    axes[-1].legend(fontsize=8)
    show(
        fig,
        "selection-nz",
        "sigma_0.003; each bin integrates to one. Target assumes constant comoving "
        "density within each source shell. This descriptive shape difference is "
        "not a fitted n(z)-sensitivity experiment.",
    )
    fig, axes = plt.subplots(1, 2, figsize=(7, 3), layout="constrained")
    labels = ["BGS" if r["tracer"].startswith("BGS") else "LRG" for r in area]
    x = np.arange(len(area))
    axes[0].bar(x - 0.18, [r["attained_fsky"] for r in area], 0.36, label="current")
    axes[0].bar(x + 0.18, [r["target_fsky"] for r in area], 0.36, label="target")
    axes[0].set(ylabel=r"$f_{\rm sky}$", xticks=x, xticklabels=labels)
    axes[0].legend(fontsize=8, loc="upper center", bbox_to_anchor=(0.5, 1.18), ncol=2)
    axes[1].bar(x, [r["sigma_factor"] for r in area], 0.55)
    axes[1].set(
        ylabel=r"Conditional $\sigma_{\rm target}/\sigma_{\rm current}$",
        xticks=x,
        xticklabels=labels,
        ylim=(0, 1.05),
    )
    show(
        fig,
        "area-projection",
        "Conditional Gaussian likelihood projection at fixed density, selection shape "
        "and reconstruction response. Uncertainty scales as area^-1/2; the 2D likelihood "
        "FoM scales as area. Relative reconstruction gain is held fixed, not forecast "
        "to improve. Priors and extra survey systematics can invalidate this scaling.",
    )


SELECTION_REPORT = None
if SELECTION_REPORT is not None:
    bundle = readers.read_tables(
        SELECTION_REPORT, ["density.json", "radial-selection.json", "area-projection.json"]
    )
    claim_figures_plot_selection(
        bundle["density.json"],
        bundle["radial-selection.json"],
        bundle["area-projection.json"],
        TIERS,
    )


In [ ]:
def lrg_figures_forecast_figures(forecasts, stats, grid):
    fig, axes = plt.subplots(1, 2, figsize=two_col(3.3), constrained_layout=True)
    for condition in grid["sigma_z_tiers"]:
        selected = [r for r in forecasts if r["condition"] == condition]
        sigma = 0 if condition == "native" else float(condition.split("_")[1])
        colour = sigma_color(sigma)
        axes[0].plot(
            [r["z_eff"] for r in selected],
            [r["sigma_r_over_r"] for r in selected],
            "o-",
            color=colour,
            ms=3,
            label=condition.replace("sigma_", ""),
        )
        axes[0].plot(
            [r["z_eff"] for r in selected],
            [r["cosmic_variance_sigma_r_over_r"] for r in selected],
            ":",
            color=colour,
            alpha=0.5,
        )
        for r in selected:
            for parameter, marker in zip(PARAMETERS, ("o", "s")):
                summary = next(
                    s
                    for s in stats
                    if (s["measurement_bin"], s["condition"], s["parameter"], s["state"])
                    == (r["measurement_bin"], condition, parameter, STATES[1])
                )
                ratio = summary["median_uncertainty_ratio"]
                if ratio is not None:
                    axes[1].scatter(
                        r["veff_mpc_h3"] / 1e9, ratio, color=colour, marker=marker, s=20
                    )
                    axes[1].annotate(
                        r["measurement_bin"] + f" ({summary['usable_pairs']}/{summary['N']})",
                        (r["veff_mpc_h3"] / 1e9, ratio),
                        fontsize=5,
                        xytext=(3, 3),
                        textcoords="offset points",
                    )
    axes[0].set(xlabel=r"$z_{\rm eff}$", ylabel=r"$\sigma_R/R$")
    axes[0].legend(fontsize=6)
    axes[1].set(
        xlabel=r"$V_{\rm eff}$ [$h^{-3}{\rm Gpc}^3$]",
        ylabel=r"Median $\sigma_{\rm post}/\sigma_{\rm pre}$",
        xscale="log",
    )
    axes[1].axhline(1, color="0.6", ls=":")
    show(
        fig,
        "density-volume-interpretation",
        "Left: existing DESI-calibrated density/volume scaling at current attained cap counts "
        "and actual footprint, in fractional units. Dotted curves are infinite-density limits "
        "for the same volumes. Right: measured profile-width ratios; circles=qiso, squares=qap; "
        "labels give usable/all pairs. The configured-P0 effective-volume proxy omits anisotropic "
        "photo-z information loss and reconstruction response. This is conditional context, "
        "not a fitted BAO forecast, a density-causation test or evidence that sky area is irrelevant.",
    )


LRG_VOLUME_REPORT = None
if LRG_VOLUME_REPORT is not None:
    volume_forecast, volume_summary, grid = readers.load_lrg_volume(LRG_VOLUME_REPORT)
    lrg_figures_forecast_figures(volume_forecast, volume_summary, grid)


## Save figures

Choose an output directory under `products/` to save PNG and PDF copies. Rerunning this cell updates those figures.


In [ ]:
# Set a directory when you want to save the plots currently in figures.
OUTPUT_DIR = None  # REPO / "products" / "figure-exploration"
if OUTPUT_DIR is not None:
    destination = Path(OUTPUT_DIR)
    destination.mkdir(parents=True, exist_ok=True)
    for name, fig in figures.items():
        fig.savefig(destination / f"{name}.png", dpi=300, bbox_inches="tight")
        fig.savefig(destination / f"{name}.pdf", bbox_inches="tight")
